### Configure PyTorch CUDA memory allocation settings to prevent fragmentation

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

### Check GPU availability and display GPU information

In [2]:
gpu_info = !nvidia-smi
gpu_info = "\n".join(gpu_info)
if gpu_info.find("failed") >= 0:
    print("Not connected to a GPU")
else:
    print(gpu_info)


Mon Mar  9 21:20:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Install required Python packages (datasets, transformers, torchaudio, jiwer, accelerate, evaluate)

In [3]:
%%capture
!pip install wget
!pip install datasets==3.6.0
!pip install transformers==4.57.1
!pip install torchaudio
!pip install jiwer
!pip install accelerate -U
!pip install evaluate


### Authenticate with Hugging Face Hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()


### Load the Sinhala ASR dataset with specified train/test splits

In [4]:
from datasets import load_dataset

common_voice_train = load_dataset(
    "keshan/large-sinhala-asr-dataset",
    "si",
    split="train[:40%]",
    trust_remote_code=True,
)

common_voice_test = load_dataset(
    "keshan/large-sinhala-asr-dataset",
    "si",
    split="test[:40%]",
    trust_remote_code=True,
)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


large-sinhala-asr-dataset.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

### Inspect the training dataset structure and size

In [5]:
common_voice_train

Dataset({
    features: ['filename', 'x', 'sentence', 'file'],
    num_rows: 53018
})

### Inspect the test dataset structure and size

In [6]:
common_voice_test

Dataset({
    features: ['filename', 'x', 'sentence', 'file'],
    num_rows: 9356
})

### Rename the 'file' column to 'audio' and drop unused columns for downstream compatibility

In [7]:
# Rename the audio-path column to match the rest of the notebook
common_voice_train = common_voice_train.rename_column("file", "audio")
common_voice_test  = common_voice_test.rename_column("file", "audio")

# Drop columns that are not used downstream (dataset-specific)
common_voice_train = common_voice_train.remove_columns(["filename", "x"])
common_voice_test  = common_voice_test.remove_columns(["filename", "x"])

from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset) - 1)
        while pick in picks:
            pick = random.randint(0, len(dataset) - 1)
        picks.append(pick)
    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

# Select the 'train' split before calling show_random_elements
show_random_elements(common_voice_train.remove_columns(["audio"]))


,sentence
0,තරුණ ප්‍රමුඛතාවන් වෙනුවෙන් පෙනී සිටින ආකාරය
1,එහි ඇතුළු නුවර හා පිට නුවර
2,හැබැයි කාර්තික්ගේ සිතුම් පැතුම් වලත්
3,ධර්මය හා ධනාත්මක චින්තනය යන
4,සිටි සෝර්බා ගේ ජීවිතය
5,එම සමාජයේ යහ පැවැත්ම උදෙසා
6,උඹටත් එකෙක් යනකොට තව එකෙක් ඉන්නවා
7,"අංගුත්තර නිකාය, සම්බෝධි වර්ගයේ"
8,ඊට සම්බන්ධ සියලුම පුද්ගලයිනුත් හෙලිකරමින් සිටී.
9,අතපය ඉදිරියට දානවා


### Define and apply text normalization: remove punctuation, Latin characters, digits, and normalize Unicode for Sinhala text

In [8]:
import re
import unicodedata

# punctuation list (kept in the same style as your notebook)
# added Sinhala/Indic danda + Sinhala punctuation marks
chars_to_remove_regex = r'[\,\?\.\!\-\;\:\"\“\%\‘\”\�\(\)\'\”\‘‘\’\ʻ\”\“\–\—\…\»\«।॥෴]'

def remove_punctuation(batch):
    text = batch["sentence"]
    text = unicodedata.normalize("NFC", text)

    # remove punctuation
    text = re.sub(chars_to_remove_regex, "", text)

    # remove latin letters + digits (Sinhala baseline)
    text = re.sub(r"[A-Za-z0-9]", "", text)

    # normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    batch["sentence"] = text
    return batch

common_voice_train = common_voice_train.map(remove_punctuation)
common_voice_test = common_voice_test.map(remove_punctuation)

# Keep only Sinhala Unicode block + spaces (Sinhala: U+0D80–U+0DFF)
sinhala_allowed = re.compile(r"^[\u0D80-\u0DFF\s]+$")

def keep_only_sinhala_rows(batch):
    s = batch["sentence"].strip()
    if len(s) == 0:
        return False
    return bool(sinhala_allowed.match(s))

common_voice_train = common_voice_train.filter(keep_only_sinhala_rows)
common_voice_test = common_voice_test.filter(keep_only_sinhala_rows)

Map:   0%|          | 0/53018 [00:00<?, ? examples/s]

Map:   0%|          | 0/9356 [00:00<?, ? examples/s]

Filter:   0%|          | 0/53018 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9356 [00:00<?, ? examples/s]

### Extract the unique character vocabulary from train and test sets

In [9]:
def extract_all_chars(batch):
    all_text = " ".join(batch["sentence"])
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/42836 [00:00<?, ? examples/s]

Map:   0%|          | 0/7580 [00:00<?, ? examples/s]

{' ': 0, 'ං': 1, 'ඃ': 2, 'අ': 3, 'ආ': 4, 'ඇ': 5, 'ඈ': 6, 'ඉ': 7, 'ඊ': 8, 'උ': 9, 'ඌ': 10, 'ඍ': 11, 'එ': 12, 'ඒ': 13, 'ඓ': 14, 'ඔ': 15, 'ඕ': 16, 'ඖ': 17, 'ක': 18, 'ඛ': 19, 'ග': 20, 'ඝ': 21, 'ඞ': 22, 'ඟ': 23, 'ච': 24, 'ඡ': 25, 'ජ': 26, 'ඣ': 27, 'ඤ': 28, 'ඥ': 29, 'ට': 30, 'ඨ': 31, 'ඩ': 32, 'ඪ': 33, 'ණ': 34, 'ඬ': 35, 'ත': 36, 'ථ': 37, 'ද': 38, 'ධ': 39, 'න': 40, 'ඳ': 41, 'ප': 42, 'ඵ': 43, 'බ': 44, 'භ': 45, 'ම': 46, 'ඹ': 47, 'ය': 48, 'ර': 49, 'ල': 50, 'ව': 51, 'ශ': 52, 'ෂ': 53, 'ස': 54, 'හ': 55, 'ළ': 56, 'ෆ': 57, '්': 58, 'ා': 59, 'ැ': 60, 'ෑ': 61, 'ි': 62, 'ී': 63, 'ු': 64, 'ූ': 65, 'ෘ': 66, 'ෙ': 67, 'ේ': 68, 'ෛ': 69, 'ො': 70, 'ෝ': 71, 'ෞ': 72, 'ෟ': 73, 'ෲ': 74, 'ෳ': 75}


### Remove all non-Sinhala characters from transcriptions and re-extract the character vocabulary

In [10]:
import re

def remove_non_sinhala_characters(batch):
    # remove anything that is NOT Sinhala block or whitespace
    batch["sentence"] = re.sub(r"[^\u0D80-\u0DFF\s]", "", batch["sentence"])
    batch["sentence"] = re.sub(r"\s+", " ", batch["sentence"]).strip()
    return batch

common_voice_train = common_voice_train.map(remove_non_sinhala_characters)
common_voice_test = common_voice_test.map(remove_non_sinhala_characters)


# extract unique characters again
vocab_train = common_voice_train.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_train.column_names,
)
vocab_test = common_voice_test.map(
    extract_all_chars,
    batched=True,
    batch_size=-1,
    keep_in_memory=True,
    remove_columns=common_voice_test.column_names,
)

vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
print(vocab_dict)


Map:   0%|          | 0/42836 [00:00<?, ? examples/s]

Map:   0%|          | 0/7580 [00:00<?, ? examples/s]

Map:   0%|          | 0/42836 [00:00<?, ? examples/s]

Map:   0%|          | 0/7580 [00:00<?, ? examples/s]

{' ': 0, 'ං': 1, 'ඃ': 2, 'අ': 3, 'ආ': 4, 'ඇ': 5, 'ඈ': 6, 'ඉ': 7, 'ඊ': 8, 'උ': 9, 'ඌ': 10, 'ඍ': 11, 'එ': 12, 'ඒ': 13, 'ඓ': 14, 'ඔ': 15, 'ඕ': 16, 'ඖ': 17, 'ක': 18, 'ඛ': 19, 'ග': 20, 'ඝ': 21, 'ඞ': 22, 'ඟ': 23, 'ච': 24, 'ඡ': 25, 'ජ': 26, 'ඣ': 27, 'ඤ': 28, 'ඥ': 29, 'ට': 30, 'ඨ': 31, 'ඩ': 32, 'ඪ': 33, 'ණ': 34, 'ඬ': 35, 'ත': 36, 'ථ': 37, 'ද': 38, 'ධ': 39, 'න': 40, 'ඳ': 41, 'ප': 42, 'ඵ': 43, 'බ': 44, 'භ': 45, 'ම': 46, 'ඹ': 47, 'ය': 48, 'ර': 49, 'ල': 50, 'ව': 51, 'ශ': 52, 'ෂ': 53, 'ස': 54, 'හ': 55, 'ළ': 56, 'ෆ': 57, '්': 58, 'ා': 59, 'ැ': 60, 'ෑ': 61, 'ි': 62, 'ී': 63, 'ු': 64, 'ූ': 65, 'ෘ': 66, 'ෙ': 67, 'ේ': 68, 'ෛ': 69, 'ො': 70, 'ෝ': 71, 'ෞ': 72, 'ෟ': 73, 'ෲ': 74, 'ෳ': 75}


### Finalize the vocabulary: replace space with pipe delimiter, add [UNK] and [PAD] tokens, and save vocab.json

In [11]:
# use "|" as word delimiter instead of space
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

# add special tokens
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print("Vocab size:", len(vocab_dict))

import json
with open("vocab.json", "w", encoding="utf-8") as vocab_file:
    json.dump(vocab_dict, vocab_file, ensure_ascii=False)


Vocab size: 78


### Initialize the CTC tokenizer, SeamlessM4T feature extractor, and Wav2Vec2-BERT processor

In [12]:
from transformers import Wav2Vec2CTCTokenizer
from transformers import SeamlessM4TFeatureExtractor
from transformers import Wav2Vec2BertProcessor

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(
    "./",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

feature_extractor = SeamlessM4TFeatureExtractor(
    feature_size=80,
    num_mel_bins=80,
    sampling_rate=16000,
    padding_value=0.0,
)

processor = Wav2Vec2BertProcessor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)


### Resample all audio to 16kHz and preview a random training sample

In [13]:
from datasets import Audio

common_voice_train = common_voice_train.cast_column("audio", Audio(sampling_rate=16_000))
common_voice_test = common_voice_test.cast_column("audio", Audio(sampling_rate=16_000))

print(common_voice_train[0]["audio"])

import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(common_voice_train) - 1)
print("Target text:", common_voice_train[rand_int]["sentence"])
print("Input array shape:", common_voice_train[rand_int]["audio"]["array"].shape)
print("Sampling rate:", common_voice_train[rand_int]["audio"]["sampling_rate"])

ipd.Audio(
    data=common_voice_train[rand_int]["audio"]["array"],
    autoplay=True,
    rate=16000,
)


{'path': '/root/.cache/huggingface/datasets/downloads/extracted/asr_sinhala/data/bd/bd535bed99.flac', 'array': array([-0.00027466,  0.00024414, -0.0007019 , ...,  0.00106812,
        0.00085449,  0.00024414]), 'sampling_rate': 16000}
Target text: කක්කුස්සි හදන්නයි
Input array shape: (75200,)
Sampling rate: 16000


### Define the dataset preparation function: extract audio features and encode text labels, then apply to train and test sets

In [14]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    ).input_features[0]
    batch["input_length"] = len(batch["input_features"])
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

common_voice_train = common_voice_train.map(
    prepare_dataset,
    remove_columns=common_voice_train.column_names,
)
common_voice_test = common_voice_test.map(
    prepare_dataset,
    remove_columns=common_voice_test.column_names,
)


Map:   0%|          | 0/42836 [00:00<?, ? examples/s]

Map:   0%|          | 0/7580 [00:00<?, ? examples/s]

### Define a custom data collator that pads input features and labels for CTC training

In [15]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union
import evaluate

from transformers import Wav2Vec2BertProcessor

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)


### Define the evaluation metrics function computing WER and CER using greedy decoding

In [16]:
import numpy as np

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    label_ids = pred.label_ids.copy()

    pred_ids = np.argmax(pred_logits, axis=-1)
    pred_str = processor.batch_decode(pred_ids)

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

### Clear GPU memory cache before loading the model

In [17]:
torch.cuda.empty_cache()

### Load the pre-trained W2V-BERT 2.0 model with an adapter and CTC head for fine-tuning

In [18]:
from transformers import Wav2Vec2BertForCTC

model = Wav2Vec2BertForCTC.from_pretrained(
    "facebook/w2v-bert-2.0",
    add_adapter=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)
print("Model loaded successfully!")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

Some weights of Wav2Vec2BertForCTC were not initialized from the model checkpoint at facebook/w2v-bert-2.0 and are newly initialized: ['lm_head.bias', 'lm_head.weight', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.residual_conv.bias', 'wav2vec2_bert.adapter.layers.0.residual_conv.weight', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.weight', 'wav2vec2_ber

Model loaded successfully!


### Configure training hyperparameters: batch size, learning rate, evaluation strategy, checkpointing, and early stopping

In [19]:
from transformers import TrainingArguments, Trainer, Wav2Vec2BertForCTC, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./outputs",
    group_by_length=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    fp16=True,

    eval_strategy="steps",     # FIXED
    eval_steps=300,

    logging_strategy="steps",
    logging_steps=300,

    save_strategy="steps",
    save_steps=3600,

    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_wer",   # FIXED
    greater_is_better=False,

    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=common_voice_train,
    eval_dataset=common_voice_test,
    tokenizer=processor
)


/tmp/ipykernel_620/928566529.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Start the fine-tuning training run

In [20]:
trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 79, 'bos_token_id': 78}.


Step,Training Loss,Validation Loss,Wer,Cer
300,2441.394400,664.570862,0.999145,0.968046
600,1591.593300,642.622437,1.068335,0.836749
900,1410.849100,621.777771,1.207325,0.818852
1200,1282.418200,607.053284,1.001343,0.861853
1500,907.720700,151.113419,0.697360,0.188911
1800,398.421700,91.703857,0.486647,0.114282
2100,316.733400,73.490768,0.429727,0.096539
2400,299.476100,63.646526,0.397131,0.087777
2700,249.539300,58.942272,0.371036,0.080309
3000,250.742000,55.131695,0.363009,0.076953


Step,Training Loss,Validation Loss,Wer,Cer
300,2441.394400,664.570862,0.999145,0.968046
600,1591.593300,642.622437,1.068335,0.836749
900,1410.849100,621.777771,1.207325,0.818852
1200,1282.418200,607.053284,1.001343,0.861853
1500,907.720700,151.113419,0.697360,0.188911
1800,398.421700,91.703857,0.486647,0.114282
2100,316.733400,73.490768,0.429727,0.096539
2400,299.476100,63.646526,0.397131,0.087777
2700,249.539300,58.942272,0.371036,0.080309
3000,250.742000,55.131695,0.363009,0.076953


TrainOutput(global_step=13390, training_loss=344.05008722694174, metrics={'train_runtime': 32821.0779, 'train_samples_per_second': 13.051, 'train_steps_per_second': 0.408, 'total_flos': 5.428770632999179e+19, 'train_loss': 344.05008722694174, 'epoch': 10.0})

### Mount Google Drive for persistent storage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Run greedy CTC inference on the test set and compute WER/CER

In [21]:
from transformers import AutoModelForCTC, Wav2Vec2BertProcessor
import torch
import numpy as np


all_pred = []
all_refs = []

checkpoint_dir = "./outputs/checkpoint-13390"

model = AutoModelForCTC.from_pretrained(checkpoint_dir).to("cuda")
processor = Wav2Vec2BertProcessor.from_pretrained(checkpoint_dir)

for ex in common_voice_test:
    input_feats = torch.tensor(ex["input_features"]).unsqueeze(0).to(model.device)

    with torch.no_grad():
        logits = model(input_feats).logits

    pred_ids = torch.argmax(logits, dim=-1)
    pred_text = processor.batch_decode(pred_ids)[0]
    all_pred.append(pred_text)

    ref_text = processor.decode(ex["labels"], group_tokens=False).lower()
    all_refs.append(ref_text)

wer = wer_metric.compute(predictions=all_pred, references=all_refs)
cer = cer_metric.compute(predictions=all_pred, references=all_refs)

print(f"WER (ASR only, greedy): {wer:.4f}")
print(f"CER (ASR only, greedy): {cer:.4f}")

for idx in range(3):
    print("\n--- Example", idx, "---")
    print("REF :", all_refs[idx])
    print("PRED:", all_pred[idx])

WER (ASR only, greedy): 0.2383
CER (ASR only, greedy): 0.0456

--- Example 0 ---
REF : ආචාර්යවරිය නම් ලකුණු කරන අතරතුර
PRED: ආචාර්යවරිය නම් ලකුණු කරන අතරතුර

--- Example 1 ---
REF : මේ අයුරින් මරණයට පත් වේ
PRED: මේ අයුරින් මරණයට පත්වේ

--- Example 2 ---
REF : ඒකට මොකද මේකෙන්
PRED: ඒකට මොකද මේකෙන්


### Mount Google Drive for persistent storage

In [22]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Copy the best checkpoint to Google Drive for persistent storage

In [24]:
!cp -r ./outputs/checkpoint-13390 /content/drive/MyDrive/final_model35